# Part II of WS4
# Deep learning: representation learning

## Context

Imagine that you installed accelerometers on a pedestrian bridge and collected thousands of hours of vibration data. The sensors work perfectly, but now you face another problem: no engineer can inspect every signal individually.

Each record mixes several causes:

- **environment:** temperature changes the bridge stiffness;
- **operation:** cyclists, runners, and other loads excite it differently;
- **structure:** damage may introduce a local change in stiffness.

![Pedestrian bridge carrying a runner, cyclist, and jumping person, with two accelerometers recording the vibration response](../../assets/bridge_monitoring_loads_v2.png)

We would like to replace every long signal with a compact representation while retaining the information that matters.

> **Can a neural network recover meaningful factors from vibration data without being told the temperature, load, or damage state?**

This problem is different from the supervised learning problem considered in the previous session. There, the neural network learned to predict a known target from labelled examples.

Here, there is no target to predict. Instead, we ask the neural network to discover useful structure directly from the data. This is known as representation learning and, in this case, falls within the unsupervised learning paradigm.
## Dataset

A real monitoring campaign rarely tells us the true temperature effect, applied force, or exact moment when damage begins. That makes it difficult to judge whether a learned representation is meaningful.

We therefore use [DynaResp](https://github.com/YacineBelHadj/DynaResp) to build a controlled virtual campaign. The simulation is not the final application; it is a laboratory in which the hidden causes are known. During training we hide those causes from the neural network, then reveal them afterward to interpret what it learned.

## Workshop flow

1. simulate the monitoring campaign
2. inspect responses and modal frequencies
3. train a denoising spectral autoencoder
4. inspect its representation
5. discuss what the representation retains 

In [ ]:
# 0. Setup and configuration

from pathlib import Path

# Scientific computing and visualization
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Structural dynamics simulation
import dynaresp as dr

# Deep learning
import lightning as L
import torch
import torch.nn as nn
import torch.nn.functional as F
from lightning.pytorch.loggers import CSVLogger
from torch.utils.data import DataLoader, Subset

# Representation analysis
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Consistent figures and reproducible experiments
plt.style.use("seaborn-v0_8-whitegrid")
L.seed_everything(42, workers=True)

## 1. Generate a monitoring campaign

For every experiment, think of the measured response as

$
\text{vibration}=f(\text{environment},\text{operation},\text{structural health}).
$

Temperature changes Young's modulus globally. A randomly selected load changes how the beam is excited. The bridge remains healthy until a local 20% stiffness reduction is introduced near the end of the test period.

The campaign contains **4,000 experiments**:

- 3,000 healthy training records;
- 200 healthy validation records;
- 400 healthy test records;
- 400 damaged test records.

Each experiment contains two accelerometers and 4,000 time samples.

In [ ]:
N_TRAIN = 3000
N_VAL = 200
N_TEST_HEALTHY = 400
N_TEST_DAMAGE = 400
N_SAMPLE = N_TRAIN + N_VAL + N_TEST_HEALTHY + N_TEST_DAMAGE

damage_start = N_TRAIN + N_VAL + N_TEST_HEALTHY
monitoring_index = np.arange(N_SAMPLE)

In [ ]:
def stiffness_scale(temperature, reference=20, alpha=2e-2):
    return 1 - alpha * (temperature - reference)


temperature_profile = (
    20
    + 10 * np.sin(2 * np.pi * 60 * monitoring_index / N_SAMPLE)
    + 7 * np.sin(2 * np.pi * 7 * monitoring_index / N_SAMPLE)
)

In [ ]:
fig, ax = plt.subplots(ncols=2, figsize=(11, 3.5))

ax[0].plot(monitoring_index, temperature_profile)
ax[0].axvline(N_TRAIN, color="tab:green", ls="--", label="Train / validation")
ax[0].axvline(N_TRAIN + N_VAL, color="tab:red", ls="--", label="Validation / test")
ax[0].axvspan(N_SAMPLE-N_TEST_DAMAGE,N_SAMPLE, color="red",alpha=0.2,label='damaged')
ax[0].set(xlabel="Monitoring index", ylabel="Temperature [°C]")
ax[0].legend(frameon=True)

ax[1].scatter(temperature_profile, stiffness_scale(temperature_profile), s=8)
ax[1].set(xlabel="Temperature [°C]", ylabel=r"$E(T)/E_0$")

plt.tight_layout()
plt.show()

The slow horizontal axis represents monitoring time: one point is one experiment. The oscillating profile represents temperature changing throughout the campaign.

The temperature coefficient is deliberately strong so that its effect can be discovered in a short workshop. It is a teaching assumption, not a calibrated material model.

---
---

### Start with one crossing

Before generating thousands of records, we inspect one crossing. This gives physical meaning to the arrays that will later enter the neural network:

```text
moving force → structural response → accelerometer measurements
```

### A quick introduction to DynaResp

[DynaResp](https://github.com/YacineBelHadj/DynaResp) simulates the dynamic response of structural systems.

For each simulation, we define a beam, a load, sensor locations, a time axis, and the conditions of the experiment.

A **case** contains the parameters that change between experiments:

```python
case = {
    "temperature": 20.0,
    "speed": 5.0,
    "load_scale": 1.0,
    "damage_reduction": 0.0,
}
```

DynaResp passes the case to the beam and load builders and then performs the simulation:

```text
case
→ build the beam and load
→ solve the dynamic response
→ record signals and modal properties
```

Providing many cases produces a monitoring campaign containing different environmental, operational, and structural conditions.

> Change one parameter in the case and predict how the acceleration response will change before rerunning the simulation.

In [ ]:
beam_properties = {"length": 10, "density": 7850, "area": 0.20,
                    "second_moment": 7.5e-4, "damping_ratio": 0.02,
                      "n_elements": 20, "n_modes": 4}

time_axis = np.arange(0, 4, 0.001)
positions = np.linspace(0, 10, 101)

case = {"temperature": 20.0, "speed": 5.0, "load_scale": 1.0,
         "damage_reduction": 0.0}

In [ ]:
def build_beam(case):
    damage_element = beam_properties["n_elements"] // 2
    damage = {damage_element: case["damage_reduction"]} if case["damage_reduction"] else None

    return dr.Beam(
        **beam_properties,
        young_modulus=210e9 * stiffness_scale(case["temperature"]),
        stiffness_reductions=damage,
    )
def build_two_wheel_load(case):
    return dr.loads.TwoWheelLoad(
        front_load=-400 * case["load_scale"],
        rear_load=-350 * case["load_scale"],
        speed=case["speed"],
        wheel_spacing=1.0,
    )

def build_runner_load(case):
    return dr.loads.RunnerLoad(
        mass=75 * case["load_scale"],
        speed=case["speed"],
        step_frequency=2.2,
        contact_duration=0.2,
    )
sensors = [dr.sensors.Accelerometer(position=x) for x in (1, 5)]

example_record = [
    dr.Record.signal,
    dr.Record.load.values,
    dr.Record.modal_parameter.frequencies_hz,
    dr.Record.modal_parameter.mode_shapes,
    dr.Record.case_parameter,
    dr.Record.time,
    dr.Record.positions,
]

In [ ]:
example = dr.generate_dataset(
    cases=[case], system_builder=build_beam, load_builder=build_two_wheel_load,
    sensors=sensors, time=time_axis, positions=positions,
    record=example_record, seed=42,
)

runner_example = dr.generate_dataset(
    cases=[case], system_builder=build_beam, load_builder=build_runner_load,
    sensors=sensors, time=time_axis, positions=positions,
    record=example_record, seed=42,
)

sample = example[0]
runner_sample = runner_example[0]

In [ ]:
from scipy.signal import welch

wheel_force = sample.load["values"]
fs = 1 / np.diff(sample.time[:2])[0]

fig, ax = plt.subplot_mosaic(
    [["sensor_1", "welch", "load"], ["sensor_2", "welch", "load"]],
    figsize=(15, 5)
)

for i, signal in enumerate(sample.signal):
    ax[f"sensor_{i+1}"].plot(sample.time, signal)
    f, psd = welch(signal, fs=fs, nperseg=1000)
    ax["welch"].plot(f, np.log(psd), label=f"Sensor {i+1}")

ax["sensor_1"].set(title="Acceleration responses", ylabel="Sensor 1")
ax["sensor_2"].set(xlabel="Time [s]", ylabel="Sensor 2")
ax["welch"].set(title="Welch spectrum", xlabel="Frequency [Hz]", ylabel="Log power", xlim=(0, 100))
ax["welch"].legend()

im = ax["load"].pcolormesh(sample.time, sample.positions, -wheel_force, shading="auto", cmap="magma")
ax["load"].set(title="Two-wheel load", xlabel="Time [s]", ylabel="Position [m]")
plt.colorbar(im, ax=ax["load"], label="Downward force [N]")

plt.tight_layout()
plt.show()

### From the load to the measured vibration

The right panel shows the two wheel forces moving along the beam. The left panels show the resulting acceleration, while the middle panel shows their Welch spectra.

Notice that the load has already left the bridge after about 2 s, but the structure continues to vibrate.

> **Question:** What can we learn from the time signals and their spectra? Why do we observe large and small peaks? What determines the spacing of the small lobes, and why are some peaks different between sensors?

<details>
<summary><strong>Show answer</strong></summary>

The **large peaks** are associated with the natural frequencies of the beam.

The smaller lobes result from **interference between the two wheels**. Their spacing depends on the wheel speed $v$ and spacing $d$.

$$
\tau = \frac{d}{v} = \frac{1}{5} = 0.2\text{ s}.
$$

At frequency $f$, this delay introduces a phase difference

$$
\Delta\phi = 2\pi f\tau.
$$

Destructive interference occurs when

$$
2\pi f\tau = (2k+1)\pi,
$$

so consecutive minima are separated by

$$
\Delta f = \frac{1}{\tau} = \frac{v}{d} = 5\text{ Hz}.
$$

The second modal peak is almost absent for Sensor 2 because this sensor is located close to a **node of the second mode**.

After the load leaves, the beam continues to vibrate freely at its natural frequencies.

</details>

In [ ]:
runner_force = runner_sample.load["values"]
fs = 1 / np.diff(runner_sample.time[:2])[0]

fig, ax = plt.subplot_mosaic(
    [["sensor_1", "welch", "load"], ["sensor_2", "welch", "load"]],
    figsize=(15, 5)
)

for i, signal in enumerate(runner_sample.signal):
    ax[f"sensor_{i+1}"].plot(runner_sample.time, signal)
    f, psd = welch(signal, fs=fs, nperseg=1000)
    ax["welch"].plot(f, np.log(psd), label=f"Sensor {i+1}")

ax["sensor_1"].set(title="Runner acceleration responses", ylabel="Sensor 1")
ax["sensor_2"].set(xlabel="Time [s]", ylabel="Sensor 2")
ax["welch"].set(title="Welch spectrum", xlabel="Frequency [Hz]", ylabel="Log power", xlim=(0, 100))
ax["welch"].legend()

im = ax["load"].pcolormesh(
    runner_sample.time, runner_sample.positions, -runner_force,
    shading="auto", cmap="magma"
)
ax["load"].set(title="Runner load", xlabel="Time [s]", ylabel="Position [m]")
plt.colorbar(im, ax=ax["load"], label="Downward force [N]")

plt.tight_layout()
plt.show()

> **Question:** Compare the runner response with the two-wheel load. What differences do you observe in the time signals and Welch spectra? 

<details>
<summary><strong>Show answer</strong></summary>

Both loads excite the **same natural frequencies**, since the structure has not changed.

However, the excitation is different. The two-wheel load produces a more obvious interference pattern espacially at sensor 1.

The measured spectrum therefore contains information about both the **structure** and the **operational conditions**.

</details>

### Modal properties

For a uniform simply supported beam,

$
f_n = \frac{n^2\pi}{2L^2}\sqrt{\frac{EI}{\rho A}}.
$

The expected frequencies are approximately **4.98, 19.90, 44.78, and 79.60 Hz**

In [ ]:
frequencies = sample.modal["frequencies_hz"]
mode_shapes = sample.modal["mode_shapes"]
fig, ax = plt.subplots(ncols=2, figsize=(10, 3), sharey=True)

for mode in range(2):
    shape = mode_shapes[mode] / abs(mode_shapes[mode]).max()
    ax[mode].plot(sample.positions, shape)
    ax[mode].axhline(0, color="black", lw=1)
    ax[mode].set(title=f"Mode {mode + 1}: {frequencies[mode]:.2f} Hz", xlabel="Position [m]")

ax[0].set_ylabel("Normalized mode shape")
plt.tight_layout()
plt.show()

> **Pause:** Which changes the beam's natural frequencies: the applied load, temperature, or local damage? Which changes only the measured response?

### Monitoring campaign

We now repeat the experiment under changing temperature and randomly varying loads. The first 3,600 experiments are healthy; damage appears only in the final 400.

This chronological ordering matters. It mimics monitoring a bridge whose future condition is unknown, rather than randomly mixing damaged observations into training.

The dataset is stored as float32 Zarr data. If it already exists, it is opened instead of regenerated. 
THe processing of generating the data takes about 6 min, but normally it is saved and should be loaded directly

In [ ]:
rng = np.random.default_rng(42)
cases = []

for index, temperature in enumerate(temperature_profile):
    load_id = rng.integers(3)
    damaged = index >= damage_start

    cases.append({
        "temperature": temperature,
        "load_id": load_id,
        "load_scale": rng.uniform(.8, 1.2),
        "speed": rng.uniform(4, 7) if load_id == 0 else rng.uniform(2, 5) if load_id == 1 else 0.0,
        "load_position": rng.uniform(2, 8),
        "step_frequency": rng.uniform(1.8, 3),
        "harmonic_frequency": rng.uniform(1, 8),
        "phase": rng.uniform(0, 2 * np.pi),
        "damage_state": int(damaged),
        "damage_reduction": .20 if damaged else 0.0,
        "split_id": 0 if index < N_TRAIN else 1 if index < N_TRAIN + N_VAL else 2,
    })

In [ ]:
def build_load(case):
    scale = case["load_scale"]

    if case["load_id"] == 0:
        return dr.loads.TwoWheelLoad(
            front_load=-4000 * scale, rear_load=-4500 * scale,
            speed=case["speed"], wheel_spacing=1,
        )
    if case["load_id"] == 1:
        return dr.loads.RunnerLoad(
            mass=75 * scale, speed=case["speed"],
            step_frequency=case["step_frequency"], contact_duration=.2,
        )
    return dr.loads.Harmonic(
        amplitude=-300 * scale,
        frequency=case["harmonic_frequency"],
        position=case["load_position"],
        phase=case["phase"],
    )

In [ ]:
record = [
    dr.Record.signal,
    dr.Record.modal_parameter.frequencies_hz,
    dr.Record.case_parameter,
    dr.Record.time,
    dr.Record.positions,
    dr.Record.metadata.seed,
    dr.Record.metadata.generation,
]

data_path = Path("data/dynaresp_bridge_harmonic_2sensors_float32.zarr")
data_path.parent.mkdir(exist_ok=True)

In [ ]:
if data_path.exists():
    dataset = dr.StructuralDataset.open(data_path)
else:
    dataset = dr.generate_dataset(
        cases=cases, system_builder=build_beam, load_builder=build_load,
        sensors=sensors, time=time_axis, positions=positions,
        record=record, storage_dtype=np.float32, output=data_path,
        seed=42, workers=4, progress=True,
    )

print(dataset.summary())

In [ ]:
properties = dataset.properties()
properties["load"] = properties.load_id.map({0: "two-wheel", 1: "runner", 2: "harmonic"})
properties["split"] = properties.split_id.map({0: "train", 1: "validation", 2: "test"})

display(
    properties.groupby(["split", "damage_state"])
    .size().unstack(fill_value=0)
)

In [ ]:
fig, ax = plt.subplots(nrows=3, figsize=(10, 6), sharex=True)

for load_id, axis in enumerate(ax):
    index = properties.query("load_id == @load_id").index[0]
    observation = dataset[index]
    axis.plot(observation.time, observation.signal.T, lw=.7)
    axis.set_ylabel(properties.loc[index, "load"])

ax[-1].set_xlabel("Time [s]")
plt.tight_layout()
plt.show()

> We can see that each type of load produces a distinct signature in the measured vibration response.

## 2. Start with what structural dynamics already gives us

Before using deep learning, let's inspect the natural frequencies.

Temperature and damage affect the structural properties and therefore the modal frequencies. The applied load changes the measured response, but not the natural frequencies.

> Can we separate these effects from the measurements alone?

In [ ]:
healthy = properties.damage_state.eq(0)
fig, ax = plt.subplots(ncols=3, figsize=(16, 4))

for load in properties.load.unique():
    rows = healthy & properties.load.eq(load)
    ax[0].scatter(properties.loc[rows, "temperature"], properties.loc[rows, "f1_hz"], s=8, alpha=.5, label=load)
ax[0].set(xlabel="Temperature [°C]", ylabel="First frequency [Hz]")
ax[0].legend()

ax[1].plot(properties.index, properties.f1_hz)
ax[1].axvline(damage_start, color="red", ls="--", label="Damage")
ax[1].set(xlabel="Monitoring index", ylabel="First frequency [Hz]")
ax[1].legend()

for state, color, label in [(0, "tab:blue", "Healthy"), (1, "tab:red", "Damaged")]:
    rows = properties.damage_state.eq(state)
    ax[2].scatter(properties.loc[rows, "f1_hz"], properties.loc[rows, "f2_hz"], color=color, s=12, alpha=.6, label=label)
ax[2].set(xlabel="First frequency [Hz]", ylabel="Second frequency [Hz]")
ax[2].legend()
plt.tight_layout()
plt.show()

### Move from modal properties to raw measurements

Modal frequencies are informative, but in real monitoring campaigns they are not directly available. They must first be estimated from measured vibration signals. So maybe Damage detection in this case is not that obvious

We now ask a different question: can an autoencoder discover useful structure directly from the acceleration measurements, without receiving temperature, load, or damage labels?

## 3. Relying only on raw measurements

Representation learning assumes that a high-dimensional observation $\mathbf{x}$ is produced by a smaller set of hidden factors $\mathbf{z}$. Here, these factors might contain information about temperature, load characteristics, and structural condition.

To learn such a representation, we use an **autoencoder**. An autoencoder consists of an **encoder**, which compresses the input into a low-dimensional latent representation $\mathbf{z}$, and a **decoder**, which reconstructs the input from this representation.

[![Autoencoder architecture](../../assets/applsci-15-06523-g001-550.jpg)](https://commons.wikimedia.org/wiki/File:Autoencoder_structure.png)

*General architecture of an autoencoder. The encoder compresses the input into a lower-dimensional latent representation, or bottleneck, while the decoder attempts to reconstruct the original input. Source:(https://www.mdpi.com/2076-3417/15/12/6523)

In our case, the network relies only on the measured acceleration. Raw acceleration enters the model, which computes a low-frequency **Welch log-power spectrum** internally. During training, mild attenuation and sensor noise create a corrupted view, while the unmodified spectrum remains the clean target:

$$
\tilde{\mathbf{x}}
\xrightarrow{\text{Welch}}
\tilde{\mathbf{s}}
\xrightarrow{\mathcal{N}_{\mathrm{train}}}
\tilde{\mathbf{s}}_N
\xrightarrow{\text{encoder}}
\mathbf{z}
\xrightarrow{\text{decoder}}
\hat{\mathbf{s}}_N
\approx
\mathcal{N}_{\mathrm{train}}(\mathbf{s}).
$$

Here, $\mathcal{N}_{\mathrm{train}}$ uses fixed statistics fitted only on the clean training spectra. For visualization, its inverse is applied to return the reconstruction to the physical log-power scale.

The network receives **no temperature, load, or damage labels**. Its task is to compress each vibration record into **16 values** and recover the clean spectral structure rather than reproduce every noisy fluctuation.

The interesting question is therefore: **what information has the network chosen to store in these 16 values?** Do they capture temperature, loading conditions, or structural changes even though the network was never explicitly told about them?

In [ ]:
class BridgeDataModule(L.LightningDataModule):
    """Prepare the bridge dataset for training, validation, and testing."""

    def __init__(self, dataset, properties, batch_size=32):
        super().__init__()
        self.dataset = dataset
        self.properties = properties
        self.batch_size = batch_size

    def setup(self, stage=None):
        # Autoencoder: the signal is both the input and the reconstruction target
        data = self.dataset.to_torch(inputs="signal", targets="signal")

        # Keep the chronological train / validation / test split defined earlier
        split = self.properties.split_id.to_numpy()
        self.train_set = Subset(data, np.where(split == 0)[0])
        self.val_set = Subset(data, np.where(split == 1)[0])
        self.test_set = Subset(data, np.where(split == 2)[0])

    def loader(self, data, shuffle=False):
        return DataLoader(data, batch_size=self.batch_size, shuffle=shuffle)

    def train_dataloader(self): return self.loader(self.train_set, True)
    def val_dataloader(self): return self.loader(self.val_set)
    def test_dataloader(self): return self.loader(self.test_set)

In [ ]:
class VibrationAugmentation(nn.Module):
    """Create a slightly corrupted version of the acceleration signal."""

    def __init__(self, gain=(0.8, 1.0), noise=(0.005, 0.03)):
        super().__init__()
        self.gain = gain
        self.noise = noise

    def forward(self, x, generator=None):
        # Randomly attenuate each vibration record
        gain = torch.rand(
            (len(x), 1, 1), device=x.device, dtype=x.dtype,
            generator=generator,
        )
        gain = self.gain[0] + gain * (self.gain[1] - self.gain[0])

        # Add sensor-dependent noise proportional to the signal RMS
        noise = torch.rand(
            (len(x), x.shape[1], 1), device=x.device, dtype=x.dtype,
            generator=generator,
        )
        noise = self.noise[0] + noise * (self.noise[1] - self.noise[0])
        rms = x.square().mean(-1, keepdim=True).sqrt().clamp_min(1e-8)

        return gain * x + noise * rms * torch.randn(
            x.shape, device=x.device, dtype=x.dtype, generator=generator,
        )


class WelchSpectrum(nn.Module):
    """Convert acceleration signals into log-power Welch spectra."""

    def __init__(
        self, sample_rate=1000.0, segment_length=1000,
        overlap=0.5, max_frequency=100.0, epsilon=1e-10,
    ):
        super().__init__()
        self.sample_rate = sample_rate
        self.segment_length = segment_length
        self.hop_length = int(segment_length * (1 - overlap))
        self.n_frequency = int(max_frequency * segment_length / sample_rate) + 1
        self.epsilon = epsilon

        # Hann window used for each Welch segment
        self.register_buffer(
            "window", torch.hann_window(segment_length), persistent=False,
        )

    def forward(self, x):
        # Split the signal into overlapping segments
        frames = x.unfold(-1, self.segment_length, self.hop_length)
        frames = frames - frames.mean(-1, keepdim=True)

        # FFT of each windowed segment
        spectrum = torch.fft.rfft(
            frames * self.window, dim=-1, norm="ortho",
        )

        # Average the power across segments and keep 0–100 Hz
        power = spectrum.abs().square().mean(-2)[..., :self.n_frequency]

        return torch.log(power + self.epsilon)

### From acceleration to a denoising task

Before entering the autoencoder, each acceleration signal goes through two steps:

1. **Augmentation:** we slightly change its amplitude and add sensor noise to create a corrupted view.
2. **Welch spectrum:** the time signal is converted into a log-power spectrum between 0 and 100 Hz.

During training, the network receives the **corrupted spectrum** but tries to reconstruct the **clean spectrum**.

$$
\text{acceleration}
\rightarrow
\text{corruption}
\rightarrow
\text{Welch spectrum}
\rightarrow
\text{autoencoder}
\rightarrow
\text{clean spectrum}
$$

> **Why corrupt the input?** We want the model to retain stable spectral information rather than simply copy every detail of the input.

In [ ]:
class SpectralConvEncoder(nn.Module):
    def __init__(self, n_sensors, n_frequency, latent_size):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(n_sensors, 32, 7, padding=3),
            nn.BatchNorm1d(32),
            nn.GELU(),
            nn.Conv1d(32, 64, 5, stride=2, padding=2),
            nn.BatchNorm1d(64),
            nn.GELU(),
            nn.Conv1d(64, 64, 3, stride=2, padding=1),
            nn.BatchNorm1d(64),
            nn.GELU(),
        )
        reduced_frequency = (n_frequency + 3) // 4
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * reduced_frequency, 128),
            nn.GELU(),
            nn.Linear(128, latent_size),
        )

    def forward(self, x):
        return self.head(self.features(x))


class DenoisingSpectralAutoencoder(L.LightningModule):
    def __init__(
        self, n_sensors, sample_rate,
        latent_size=16, learning_rate=3e-4,
    ):
        super().__init__()
        self.save_hyperparameters()

        self.augment = VibrationAugmentation()
        self.transform = WelchSpectrum(
            sample_rate=sample_rate,
            segment_length=round(sample_rate),
        )
        n_features = n_sensors * self.transform.n_frequency
        self.register_buffer(
            "spectrum_mean", torch.zeros(1, n_sensors, self.transform.n_frequency),
        )
        self.register_buffer(
            "spectrum_std", torch.ones(1, n_sensors, self.transform.n_frequency),
        )
        self.encoder = SpectralConvEncoder(
            n_sensors, self.transform.n_frequency, latent_size,
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_size, 128),
            nn.GELU(),
            nn.Linear(128, n_features),
        )

    @torch.inference_mode()
    def fit_spectrum_normalization(self, loader):
        total = torch.zeros_like(self.spectrum_mean, dtype=torch.float64)
        total_square = torch.zeros_like(total)
        count = 0

        for x, _ in loader:
            spectrum = self.transform(x.to(self.device)).double()
            total += spectrum.sum(0, keepdim=True)
            total_square += spectrum.square().sum(0, keepdim=True)
            count += len(spectrum)

        mean = total / count
        std = (total_square / count - mean.square()).clamp_min(1e-12).sqrt()
        self.spectrum_mean.copy_(mean.float())
        self.spectrum_std.copy_(std.float())

    def normalize(self, spectrum):
        return (spectrum - self.spectrum_mean) / self.spectrum_std

    def denormalize(self, spectrum):
        return spectrum * self.spectrum_std + self.spectrum_mean

    def encode(self, x):
        return self.encoder(self.normalize(self.transform(x)))

    def forward(self, x):
        reconstruction = self.decoder(self.encode(x))
        return reconstruction.reshape(
            len(x), -1, self.transform.n_frequency,
        )

    def reconstruct_spectrum(self, x):
        return self.denormalize(self(x))

    def step(self, batch, stage, batch_idx):
        x, _ = batch
        generator = None
        if stage == "val":
            generator = torch.Generator(device=x.device).manual_seed(
                10_000 + batch_idx,
            )

        clean = self.transform(x)
        corrupted = self.transform(self.augment(x, generator))
        clean = self.normalize(clean)
        corrupted = self.normalize(corrupted)
        reconstruction = self.decoder(
            self.encoder(corrupted),
        ).reshape_as(clean)
        loss = F.mse_loss(reconstruction, clean)

        self.log(
            f"{stage}_loss", loss,
            on_step=False, on_epoch=True,
            prog_bar=stage == "val", batch_size=len(x),
        )
        return loss

    def training_step(self, batch, batch_idx):
        return self.step(batch, "train", batch_idx)

    def validation_step(self, batch, batch_idx):
        return self.step(batch, "val", batch_idx)

    def configure_optimizers(self):
        return torch.optim.AdamW(
            self.parameters(),
            lr=self.hparams.learning_rate,
            weight_decay=1e-4,
        )

The cell below trains the model, however here we pretrained the model for you as it would require a lot of computation time

In [ ]:
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint

RUN_NAME = "welch_denoising_clean_norm_v1"

dm = BridgeDataModule(dataset, properties)
dm.setup()
logger = CSVLogger(
    "logs",
    name="bridge_denoising_spectral_autoencoder",
    version=RUN_NAME,
)

checkpoint_path = Path(logger.log_dir) / "checkpoints" / "best.ckpt"

if checkpoint_path.exists():
    print(f"Loading existing model: {checkpoint_path}")
    model = DenoisingSpectralAutoencoder.load_from_checkpoint(
        checkpoint_path,
        map_location="cpu",
    )

else:
    print("No checkpoint found; training model.")

    checkpoint = ModelCheckpoint(
        dirpath=checkpoint_path.parent,
        filename=checkpoint_path.stem,
        monitor="val_loss",
        mode="min",
        save_top_k=1,
    )

    model = DenoisingSpectralAutoencoder(
        n_sensors=dataset.signal.shape[1],
        sample_rate=float(1 / (time_axis[1] - time_axis[0])),
        latent_size=16,
    )
    model.fit_spectrum_normalization(dm.loader(dm.train_set))

    trainer = L.Trainer(
        max_epochs=100,
        logger=logger,
        callbacks=[
            checkpoint,
            EarlyStopping(
                monitor="val_loss",
                mode="min",
                patience=12,
            ),
        ],
        gradient_clip_val=1.0,
    )

    trainer.fit(model, datamodule=dm)

    model = DenoisingSpectralAutoencoder.load_from_checkpoint(
        checkpoint.best_model_path,
        map_location="cpu",
    )

model.eval()

> **Question:** Read the model code and describe its architecture.

<details>
<summary><strong>Show answer</strong></summary>

The model creates a corrupted view using mild common attenuation and sensor noise. It divides both acceleration signals into overlapping one-second Hann-windowed segments, removes each segment mean, and averages their spectral power. Only the 0–100 Hz log-power spectrum is retained: 101 values per sensor.

A fixed mean and standard deviation fitted from clean training records standardize every sensor-frequency bin. A one-dimensional convolutional encoder compresses the two spectra into a 16-dimensional latent vector, and a small dense decoder reconstructs the normalized clean spectrum. Learning corrupted-to-clean reconstruction encourages the embedding to retain repeatable spectral structure instead of individual noise fluctuations.

</details>

In [ ]:
history = pd.read_csv(Path(logger.log_dir) / "metrics.csv")

for name in ("train_loss", "val_loss"):
    curve = history.dropna(subset=[name]).groupby("epoch")[name].last()
    plt.plot(curve, label=name)

plt.xlabel("Epoch")
plt.ylabel("Normalized spectral MSE")
plt.legend()
plt.show()

### Training

Training and validation losses decrease together, indicating that the model learns to reconstruct unseen healthy data.

> But reconstruction performance does not tell us what is encoded in the latent space.

### Did the autoencoder learn to denoise?

The reconstruction follows the clean spectrum and preserves its main peaks.

> Good reconstruction is not enough. We now need to inspect **what the model learned to represent**.

In [ ]:
x, _ = next(iter(dm.val_dataloader()))
x = x.to(model.device)

generator = torch.Generator(device=x.device).manual_seed(42)
corrupted = model.augment(x, generator)

with torch.inference_mode():
    clean_spectrum = model.transform(x).cpu()
    corrupted_spectrum = model.transform(corrupted).cpu()
    reconstruction = model.reconstruct_spectrum(corrupted).cpu()

n_sensors = model.hparams.n_sensors
n_frequency = model.transform.n_frequency
frequency = np.fft.rfftfreq(
    model.transform.segment_length,
    d=1 / model.transform.sample_rate,
)[:n_frequency]

sample_index = dm.val_set.indices[0]
frequency_columns = [
    column for column in ("f1_hz", "f2_hz", "f3_hz", "f4_hz")
    if column in properties
]
modal_frequencies = properties.loc[
    sample_index, frequency_columns,
].to_numpy()
print(properties.loc[
    sample_index,
    ["load", "temperature", "speed", "step_frequency", "harmonic_frequency"],
])

fig, axes = plt.subplots(n_sensors, figsize=(10, 4), sharex=True)

for sensor, axis in enumerate(np.atleast_1d(axes)):
    axis.plot(frequency, clean_spectrum[0, sensor], label="Clean target")
    axis.plot(
        frequency, corrupted_spectrum[0, sensor],
        color="0.6", alpha=0.7, label="Corrupted input",
    )
    axis.plot(frequency, reconstruction[0, sensor], label="Reconstruction")
    axis.set_ylabel(f"Sensor {sensor + 1}\nlog power")

for axis in np.atleast_1d(axes):
    for index, modal_frequency in enumerate(modal_frequencies):
        axis.axvline(
            modal_frequency, color="black", linestyle=":", alpha=0.6,
            label="True modes" if index == 0 else None,
        )

axes[0].legend()
axes[-1].set(xlabel="Frequency [Hz]", xlim=(0, 100))
fig.suptitle("Denoising reconstruction from a corrupted view")
plt.tight_layout()
plt.show()

### Did the autoencoder learn to denoise?

The reconstruction follows the clean spectrum and preserves its main peaks.

> Good reconstruction is not enough. We now need to inspect **what the model learned to represent**.

## 4. Inspect the learned representation

The encoder was trained only to recover clean Welch log-power spectra from noisy, mildly attenuated acceleration records. It never received temperature, load, or damage labels.


In [ ]:
model.eval()

# here we compure the embeddings using the trained model
@torch.inference_mode()
def encode_subset(subset):
    batches = []

    for x, _ in dm.loader(subset):
        batches.append(model.encode(x.to(model.device)).cpu())

    return torch.cat(batches).numpy()


z_train = encode_subset(dm.train_set)
z_validation = encode_subset(dm.val_set)
z_test = encode_subset(dm.test_set)

train_properties = properties.iloc[dm.train_set.indices].copy()
validation_properties = properties.iloc[dm.val_set.indices].copy()
test_properties = properties.iloc[dm.test_set.indices].copy()

assert len(z_train) == len(train_properties)
assert len(z_validation) == len(validation_properties)
assert len(z_test) == len(test_properties)

print(
    f"Latent shapes: train={z_train.shape}, "
    f"validation={z_validation.shape}, test={z_test.shape}"
)

### Visualizing the latent space with t-SNE

Each vibration record is represented by a 16-dimensional latent vector $\mathbf{z}$. To visualize these representations, we use **t-SNE** to project them into two dimensions.

t-SNE is a **nonlinear dimensionality-reduction method** that tries to preserve local neighborhoods: points that are close in the original latent space tend to remain close in the 2D projection.


In [ ]:
from sklearn.manifold import TSNE # 
from sklearn.preprocessing import StandardScaler

z = np.vstack([z_train, z_validation, z_test])
meta = pd.concat([train_properties, validation_properties, test_properties],
                 ignore_index=True)

z_scaled = StandardScaler().fit(z_train).transform(z)
z_2d = TSNE(perplexity=30, learning_rate="auto", init="random",
            random_state=42).fit_transform(z_scaled)

views = [
    (meta.load_id, "Load: 0=wheel, 1=runner, 2=harmonic", "tab10"),
    (meta.temperature, "Temperature [°C]", "coolwarm"),
    (meta.damage_state, "State: 0=healthy, 1=damaged", "coolwarm"),
]

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharex=True, sharey=True)

for ax, (values, title, cmap) in zip(axes, views):
    points = ax.scatter(*z_2d.T, c=values, cmap=cmap, s=8, alpha=.6)
    fig.colorbar(points, ax=ax)
    ax.set(title=title, xlabel="t-SNE 1")

axes[0].set_ylabel("t-SNE 2")
plt.tight_layout()
plt.show()

## 4. What did the model learn?

Each vibration record is now represented by only 16 numbers. We use t-SNE to visualize these representations in 2D, with the points colored by load, temperature, and damage.

> If the model was never given these labels, which of them can we recover from the latent space?

In a real monitoring campaign, these labels would not be available. We would have to interpret the different regions ourselves, for example by inspecting the signals or spectra associated with each cluster. An example of this approach can be found in this [link](https://molab.marimo.io/github/koaning/notebooks/blob/main/evoc-fashion.py/server).

Here, t-SNE is only a visualization tool: it reduces the 16-dimensional representation to 2D while trying to preserve local neighborhoods. The resulting axes have no physical meaning.